# 29 · Every model on one set

Every notebook trains on notebook 0's training split, picks its best epoch on the select split, and scores its models on
the validation split the same way (`training.val_scores`: SSIM-Y, and luma and chroma PSNR, of each patch's 128 px
core), keeping each model's per-patch scores with `training.record`. This notebook sets them all side by side: one
table of every model of every notebook, against notebook 1's bicubic, the skip every model is built on.

- **The 95% range** of a gain is a bootstrap over the validation sources: the patches of one source move together.
- **Seeds.** Where a notebook trained a model at several seeds, each seed is its own row: how far apart they land is
  the luck in any one row.
- **The graphs.** Every notebook checks its exported graph against its torch model before it publishes it, and
  notebooks 3, 23 and 24 score their graphs on the validation split too, so a row here is also what the graph scores.

Needs: every notebook that records scores (1–12, 16, 18, 20, 22, 24, 26, 27), run first.

In [1]:
"""Settings."""

import os

import numpy as np

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import training

manifest = training.load_manifest()
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]
results = {f"{notebook:02d} · {name}": scores for (notebook, name), scores in sorted(recorded.items())}
print(f"{len(results)} models from {len({notebook for notebook, _ in recorded})} notebooks, "
      f"{len(bicubic['ssim']):,} validation patches each")

78 models from 17 notebooks, 24,286 validation patches each


## 1. Every model

Best first: SSIM-Y, its gain over bicubic with the 95% range, luma PSNR and its gain, chroma PSNR's gain, and the
SSIM-Y gain by what a patch shows and by the host's line.

In [2]:
training.show(training.compare(results, manifest, base=bicubic))

                                        model SSIM-Y over bicubic          95% range luma dB dB over chroma dB over   video  screen   photo    x264   nvenc
             20 · told the QP, split at QP 30 0.8926      +0.0405 +0.0381 to +0.0429   30.31   +1.72          +0.05 +0.0236 +0.0595 +0.0259 +0.0428 +0.0382
     20 · told the QP, split at QP 30, seed 1 0.8924      +0.0403 +0.0380 to +0.0428   30.30   +1.72          +0.05 +0.0233 +0.0594 +0.0257 +0.0426 +0.0381
              27 · told 24x2, mix 0.1, seed 1 0.8922      +0.0401 +0.0378 to +0.0424   30.34   +1.76          +0.04 +0.0236 +0.0582 +0.0269 +0.0421 +0.0381
           22 · told QP, loop filter, I frame 0.8919      +0.0398 +0.0375 to +0.0422   30.20   +1.62          +0.05 +0.0232 +0.0582 +0.0262 +0.0419 +0.0378
                      27 · told 24x2, mix 0.1 0.8918      +0.0397 +0.0375 to +0.0421   30.35   +1.77          +0.05 +0.0235 +0.0578 +0.0265 +0.0418 +0.0378
   22 · told QP, loop filter, I frame, seed 1 0.8915      +0.039

## 2. By slice QP

The SSIM-Y gain over bicubic in each slice QP bin; the first row counts the patches.

In [3]:
training.show(training.gain_by_qp(results, manifest, base=bicubic))

                                        model    0-18   18-26   26-30   30-34   34-38   38-42   42-46   46-52
                                      patches    3610    3928    3730    3585    3539    2846    2005    1043
                          01 · Lanczos-3, PIL +0.0023 +0.0016 +0.0010 +0.0005 +0.0000 -0.0004 -0.0005 -0.0006
                                 01 · bicubic +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000
                 01 · bicubic, PIL (a = -0.5) -0.0020 -0.0016 -0.0011 -0.0007 -0.0003 +0.0002 +0.0004 +0.0007
                                01 · bilinear -0.0175 -0.0144 -0.0114 -0.0080 -0.0048 -0.0020 +0.0001 +0.0017
                                 01 · nearest -0.0156 -0.0166 -0.0173 -0.0172 -0.0166 -0.0149 -0.0126 -0.0107
                   02 · plain CNN 16-24-24-32 +0.0298 +0.0338 +0.0368 +0.0377 +0.0372 +0.0374 +0.0305 +0.0268
                         02 · plain CNN 64-32 +0.0305 +0.0344 +0.0372 +0.0382 +0.0378 +0.0380 +0.0309 +0.0268
          

## 3. What it says

- **78 models from 17 notebooks, and the body told its stream leads.** Every run of the luma 24 + 2-block body told
  its stream (notebooks 8, 16, 18, 20, 22 and 27) is above every blind model, except those trained on the JND loss or
  on mix 1: +0.0375 to +0.0405 SSIM-Y over bicubic, against +0.0362 to +0.0365 for the same body blind (three seeds).
- **At the top, three methods lie within 0.0011.** The six best rows (0.8915 to 0.8926) are the two told models
  switched at QP 30 (notebook 20), mix 0.1 (notebook 27) and the model told the QP, loop filter and frame type
  (notebook 22), two seeds each, whose seeds land 0.0002 to 0.0004 apart. Their 95% ranges all overlap. The switched
  pair is first at both seeds (+0.0405 and +0.0403), +0.0008 over notebook 22's model; mix 0.1 and notebook 22's model
  are level.
- **Down the board**: the plain CNN 64-32 +0.0351 (at 26,688 multiply-adds and 3.05 dB of chroma lost); the fine-tuned
  `SDY x2` table +0.0337; the 32-16 family +0.0283 to +0.0315; DirNet full +0.0290; the fast q32 +0.0264; the
  published fast q24 +0.0236 and +0.0238 (two seeds); the DCT models that decide +0.011 to +0.016; every static filter
  within 0.001 of bicubic, but bilinear and nearest, below it. One seed of notebook 3's residual 32-16 never left
  bicubic (+0.0002).
- **The told models gain most at QP 38 to 42 and lead the blind one most at the ends.** Notebook 22's model gains
  +0.0439 at QP 38–42 and +0.0341 at 46–52; its lead over the blind body is +0.0053 at QP 0–18 and +0.0060 at 46–52,
  but +0.0015 at 30–34. The QP 38+ specialist (notebook 18) is the best row of all at QP 38–42 (+0.0451) and below
  bicubic at 0–18 (−0.0013).
- **The JND loss gains least where the stream is worst.** The told 24×2 trained on it gains +0.0253 to +0.0257 SSIM-Y
  at QP 0–18 and +0.0158 to +0.0165 at 46–52; trained on the SSIM loss it gains the most at 38–42. (SSIM-Y is a judge
  the JND loss does not train on; notebook 26 has the others.)
- **Colour stays bicubic's** in every luma model, within 0.05 dB of chroma PSNR, except the fast q20, q24 and q32 with
  their skip trained (−0.91 to −1.03 dB). The RGB models of notebooks 2, 3 and 6 lose 2.35 to 3.66 dB.

**Not measured here:** speed, and every judge but SSIM-Y and PSNR (LPIPS, GMSD and JND are in notebooks 26 and 27).